# 🧹 Limpieza y Estructuración de Datos — Justicia Espacial CDMX

**Proyecto:** Índice de Injusticia en el Trayecto a Centros de Salud Pública  
**Pipeline:** Bronze → Silver → Gold  
**Fuentes:** FGJ 2024 · C5 WiFi · GTFS Stops · INEGI RI-6  

---

## Contenido
1. [Dependencias y configuración](#1)
2. [FGJ — Carpetas de investigación 2024](#2)
3. [C5 WiFi — Postes de infraestructura](#3)
4. [GTFS — Paradas de transporte público](#4)
5. [RI-6 — Infraestructura física por colonia](#5)
6. [Unificación y exportación Gold Layer](#6)


## 1. Dependencias y Configuración <a id='1'></a>

In [ ]:
# Instalación (ejecutar solo si es necesario)
# !pip install geopandas duckdb pyarrow openpyxl shapely

import pandas as pd
import geopandas as gpd
import numpy as np
import duckdb
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f'pandas     {pd.__version__}')
print(f'geopandas  {gpd.__version__}')
print(f'duckdb     {duckdb.__version__}')
print(f'numpy      {np.__version__}')


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Ajusta BRONZE al directorio donde tengas los archivos originales
BRONZE = Path('data/bronze')
SILVER = Path('data/silver'); SILVER.mkdir(parents=True, exist_ok=True)
GOLD   = Path('data/gold');   GOLD.mkdir(parents=True, exist_ok=True)

FGJ_PATH   = BRONZE / 'carpetasFGJ_2024.csv'
WIFI_PATH  = BRONZE / '07-2025-wifi_gratuito_en_postes-del-c5.xlsx'
STOPS_PATH = BRONZE / 'stops.txt'
INFRA_PATH = BRONZE / 'infra_fisica/ri_6.shp'

# ── Proyecciones ────────────────────────────────────────────────────────────
WGS84 = 'EPSG:4326'    # latitud/longitud geografica
UTM14 = 'EPSG:32614'   # metros — calculos espaciales en CDMX

# ── Bounding box CDMX ───────────────────────────────────────────────────────
BBOX_CDMX = {
    'lat_min': 19.05, 'lat_max': 19.65,
    'lon_min': -99.40, 'lon_max': -98.93
}

# ── Diccionario de normalizacion de alcaldias ───────────────────────────────
ALCALDIAS_MAP = {
    'ALVARO OBREGON'         : 'Alvaro Obregon',
    'Alvaro Obregon'         : 'Alvaro Obregon',
    'AZCAPOTZALCO'           : 'Azcapotzalco',
    'BENITO JUAREZ'          : 'Benito Juarez',
    'Benito Juarez'          : 'Benito Juarez',
    'COYOACAN'               : 'Coyoacan',
    'Coyoacan'               : 'Coyoacan',
    'CUAJIMALPA DE MORELOS'  : 'Cuajimalpa',
    'Cuajimalpa de Morelos'  : 'Cuajimalpa',
    'CUAUHTEMOC'             : 'Cuauhtemoc',
    'Cuauhtemoc'             : 'Cuauhtemoc',
    'GUSTAVO A. MADERO'      : 'Gustavo A. Madero',
    'IZTACALCO'              : 'Iztacalco',
    'IZTAPALAPA'             : 'Iztapalapa',
    'LA MAGDALENA CONTRERAS' : 'Magdalena Contreras',
    'Magdalena Contreras'    : 'Magdalena Contreras',
    'MIGUEL HIDALGO'         : 'Miguel Hidalgo',
    'MILPA ALTA'             : 'Milpa Alta',
    'TLAHUAC'                : 'Tlahuac',
    'Tlahuac'                : 'Tlahuac',
    'TLALPAN'                : 'Tlalpan',
    'VENUSTIANO CARRANZA'    : 'Venustiano Carranza',
    'XOCHIMILCO'             : 'Xochimilco',
}

print('Configuracion cargada')
print(f'  Bronze: {BRONZE}')
print(f'  Silver: {SILVER}')
print(f'  Gold:   {GOLD}')


---
## 2. FGJ — Carpetas de Investigación 2024 <a id='2'></a>

**Archivo:** `carpetasFGJ_2024.csv`  
**Registros:** 138,630 filas · 21 columnas  

### Problemas detectados
| Campo | Problema | Acción |
|---|---|---|
| `latitud` / `longitud` | 8,997 nulos (6.5%) | Descartar para análisis espacial |
| `colonia_hecho` | 8,992 nulos (6.5%) | Mantener, no imputable |
| `alcaldia_hecho` | 15+ variantes del mismo nombre | Normalizar con diccionario |
| `fecha_inicio` / `fecha_hecho` | Tipo `str`, no `datetime` | Convertir con `pd.to_datetime` |

> ⚠️ **Cifra negra estimada ~85%.** Los delitos aquí registrados son solo los reportados.

In [ ]:
# ── 2.1 Carga y auditoria inicial ─────────────────────────────────────────
df_fgj = pd.read_csv(FGJ_PATH, dtype_backend='pyarrow')

print(f'Shape original: {df_fgj.shape}')
print(f'Duplicados exactos: {df_fgj.duplicated().sum()}')
print()
nulls = df_fgj.isnull().sum()
print('Columnas con nulos:')
for col, n in nulls[nulls > 0].items():
    pct = n / len(df_fgj) * 100
    flag = '⚠️ ' if pct > 5 else ('!  ' if pct > 1 else '·  ')
    print(f'  {flag}{col:<28} {n:>6} nulos  ({pct:.1f}%)')


In [ ]:
# ── 2.2 Exploracion de categorias ─────────────────────────────────────────
print('Categorias de delito:')
print(df_fgj['categoria_delito'].value_counts().to_string())
print()
print(f'Tipos unicos de delito: {df_fgj["delito"].nunique()}')


In [ ]:
# ── 2.3 Limpieza ──────────────────────────────────────────────────────────
df_clean = df_fgj.copy()

# 1. Convertir fechas a datetime
for col in ['fecha_inicio', 'fecha_hecho']:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# 2. Normalizar alcaldias
df_clean['alcaldia_norm'] = (
    df_clean['alcaldia_hecho']
    .str.strip().str.upper()
    .map(ALCALDIAS_MAP)
    .fillna(df_clean['alcaldia_hecho'])
)

# 3. Hora limpia HH:MM
df_clean['hora_inicio_clean'] = df_clean['hora_inicio'].str[:5]

# 4. Mes como numero ordinal
MESES_NUM = {
    'Enero':1, 'Febrero':2, 'Marzo':3, 'Abril':4,
    'Mayo':5, 'Junio':6, 'Julio':7, 'Agosto':8,
    'Septiembre':9, 'Octubre':10, 'Noviembre':11, 'Diciembre':12
}
df_clean['mes_num'] = df_clean['mes_inicio'].map(MESES_NUM)

# 5. Descartar sin coordenadas
n_sin_coords = df_clean['latitud'].isna().sum()
df_geo = df_clean.dropna(subset=['latitud', 'longitud']).copy()
print(f'Descartados sin coords: {n_sin_coords:,} ({n_sin_coords/len(df_clean)*100:.1f}%)')
print(f'Registros con coords:   {len(df_geo):,}')


In [ ]:
# ── 2.4 Verificar bbox CDMX ───────────────────────────────────────────────
fuera = df_geo[
    (df_geo.latitud  < BBOX_CDMX['lat_min']) | (df_geo.latitud  > BBOX_CDMX['lat_max']) |
    (df_geo.longitud < BBOX_CDMX['lon_min']) | (df_geo.longitud > BBOX_CDMX['lon_max'])
]
print(f'Registros fuera de bbox CDMX: {len(fuera)}')
if len(fuera) > 0:
    print(fuera[['delito', 'alcaldia_hecho', 'latitud', 'longitud']].head())


In [ ]:
# ── 2.5 Clasificar delitos por relevancia ─────────────────────────────────
def clasificar_delito(delito):
    d = str(delito).upper().strip()
    if any(p in d for p in [
        'TRANSEUNTE', 'LESIONES INTENCIONAL',
        'HOMICIDIO DOLOSO', 'VIOLACION', 'DISPARO DE ARMA'
    ]):
        return 'peaton'
    if any(p in d for p in ['PASAJERO', 'METRO', 'MICROBUS', 'TAXI']):
        return 'transporte'
    return 'otro'

df_geo['relevancia'] = df_geo['delito'].apply(clasificar_delito)

print('Distribucion por relevancia:')
resumen = df_geo['relevancia'].value_counts()
for cat, n in resumen.items():
    print(f'  {cat:<12}: {n:>7,}  ({n/len(df_geo)*100:.1f}%)')


In [ ]:
# ── 2.6 Crear GeoDataFrame y exportar Silver ──────────────────────────────
COLS_SILVER_FGJ = [
    'fecha_inicio', 'fecha_hecho', 'hora_inicio_clean', 'mes_num',
    'delito', 'categoria_delito', 'relevancia',
    'alcaldia_norm', 'colonia_hecho',
    'latitud', 'longitud',
]

gdf_fgj = gpd.GeoDataFrame(
    df_geo[COLS_SILVER_FGJ],
    geometry=gpd.points_from_xy(df_geo.longitud, df_geo.latitud),
    crs=WGS84
).to_crs(UTM14)

gdf_fgj.to_parquet(SILVER / 'fgj_2024.parquet')
print(f'Silver FGJ exportado')
print(f'  Filas:    {len(gdf_fgj):,}')
print(f'  Columnas: {gdf_fgj.shape[1]}')
print(f'  CRS:      {gdf_fgj.crs.name}')
gdf_fgj.drop(columns='geometry').head(3)


---
## 3. C5 WiFi — Postes de Infraestructura Digital <a id='3'></a>

**Archivo:** `07-2025-wifi_gratuito_en_postes-del-c5.xlsx`  
**Registros:** 13,714 filas · 5 columnas · **0 nulos**  

Dataset más limpio de los cuatro. Único problema: nombres de alcaldías sin acentos (`Alvaro Obregon`, `Tlahuac`) y `Cuajimalpa de Morelos` en lugar del nombre corto.  

> **Uso en el índice:** densidad de postes C5 = proxy de inversión institucional en infraestructura urbana.

In [ ]:
# ── 3.1 Carga y auditoria ─────────────────────────────────────────────────
df_wifi = pd.read_excel(WIFI_PATH, dtype_backend='pyarrow')

print(f'Shape: {df_wifi.shape}')
print(f'Nulos: {df_wifi.isnull().sum().to_dict()}')
print(f'Duplicados: {df_wifi.duplicated().sum()}')
print(f'Programa unico: {df_wifi["programa"].unique().tolist()}')
print(f'Rango lat: [{df_wifi.latitud.min():.4f}, {df_wifi.latitud.max():.4f}]')
print(f'Rango lon: [{df_wifi.longitud.min():.4f}, {df_wifi.longitud.max():.4f}]')
print()
print('Distribucion por alcaldia (campo original):')
print(df_wifi['alcaldia'].value_counts().to_string())


In [ ]:
# ── 3.2 Limpieza y exportacion Silver ────────────────────────────────────
df_wifi_clean = df_wifi.copy()

# Normalizar alcaldias
df_wifi_clean['alcaldia_norm'] = (
    df_wifi_clean['alcaldia'].map(ALCALDIAS_MAP)
    .fillna(df_wifi_clean['alcaldia'])
)

# Renombrar id para evitar colisiones en joins
df_wifi_clean = df_wifi_clean.rename(columns={'id': 'poste_id'})

# Verificar bbox
fuera = df_wifi_clean[
    (df_wifi_clean.latitud  < BBOX_CDMX['lat_min']) | (df_wifi_clean.latitud  > BBOX_CDMX['lat_max']) |
    (df_wifi_clean.longitud < BBOX_CDMX['lon_min']) | (df_wifi_clean.longitud > BBOX_CDMX['lon_max'])
]
print(f'Postes fuera de bbox CDMX: {len(fuera)}')

# GeoDataFrame
gdf_wifi = gpd.GeoDataFrame(
    df_wifi_clean[['poste_id', 'alcaldia_norm', 'latitud', 'longitud']],
    geometry=gpd.points_from_xy(df_wifi_clean.longitud, df_wifi_clean.latitud),
    crs=WGS84
).to_crs(UTM14)

gdf_wifi.to_parquet(SILVER / 'wifi_c5.parquet')
print(f'Silver WiFi exportado: {len(gdf_wifi):,} postes')
print()
print('Distribucion normalizada por alcaldia:')
print(gdf_wifi['alcaldia_norm'].value_counts().to_string())


---
## 4. GTFS — Paradas de Transporte Público <a id='4'></a>

**Archivo:** `stops.txt`  
**Registros:** 11,362 filas · 6 columnas · **0 nulos**  

| Campo | Descripción | Problema |
|---|---|---|
| `zone_id` | Codifica ruta/sistema | Decodificar sistema de transporte |
| `wheelchair_boarding` | 0=sin info, 1=accesible, 2=no accesible | Valores poco legibles |
| `stop_lat/lon` | Coordenadas | 47 paradas fuera del bbox CDMX |


In [ ]:
# ── 4.1 Carga y auditoria ─────────────────────────────────────────────────
df_stops = pd.read_csv(STOPS_PATH)

print(f'Shape: {df_stops.shape}')
print(f'Columnas: {df_stops.columns.tolist()}')
print(f'Nulos: {df_stops.isnull().sum().to_dict()}')
print(f'Duplicados stop_id: {df_stops["stop_id"].duplicated().sum()}')
print()
print('wheelchair_boarding valores:')
print(df_stops['wheelchair_boarding'].value_counts().to_string())
print()
print('Ejemplos stop_id:')
print(df_stops['stop_id'].head(6).tolist())

fuera = df_stops[
    (df_stops.stop_lat < BBOX_CDMX['lat_min']) | (df_stops.stop_lat > BBOX_CDMX['lat_max']) |
    (df_stops.stop_lon < BBOX_CDMX['lon_min']) | (df_stops.stop_lon > BBOX_CDMX['lon_max'])
]
print(f'Paradas fuera de bbox CDMX: {len(fuera)}')
if len(fuera) > 0:
    print(fuera[['stop_id','stop_name','stop_lat','stop_lon']].head())


In [ ]:
# ── 4.2 Limpieza y exportacion Silver ────────────────────────────────────
df_stops_clean = df_stops.copy()

# Inferir sistema de transporte desde zone_id
def inferir_sistema(zone_id):
    z = str(zone_id).upper()
    if z.startswith('0300'): return 'Metrobus'
    if z.startswith('05'):   return 'Metro'
    if z.startswith('04'):   return 'RTP'
    if z.startswith('06'):   return 'Tren Ligero'
    if z.startswith('07'):   return 'Cablebus'
    if z.startswith('08'):   return 'Trolebus'
    return 'Otro'

df_stops_clean['sistema'] = df_stops_clean['zone_id'].apply(inferir_sistema)

# Accesibilidad legible
df_stops_clean['accesible'] = df_stops_clean['wheelchair_boarding'].map({
    0: 'sin_info', 1: 'accesible', 2: 'inaccesible'
})

# Filtrar paradas fuera de CDMX
n_orig = len(df_stops_clean)
df_stops_clean = df_stops_clean[
    (df_stops_clean.stop_lat.between(BBOX_CDMX['lat_min'], BBOX_CDMX['lat_max'])) &
    (df_stops_clean.stop_lon.between(BBOX_CDMX['lon_min'], BBOX_CDMX['lon_max']))
].copy()
print(f'Paradas eliminadas por bbox: {n_orig - len(df_stops_clean)}')

# GeoDataFrame
gdf_stops = gpd.GeoDataFrame(
    df_stops_clean[['stop_id', 'stop_name', 'sistema', 'accesible', 'stop_lat', 'stop_lon']],
    geometry=gpd.points_from_xy(df_stops_clean.stop_lon, df_stops_clean.stop_lat),
    crs=WGS84
).to_crs(UTM14)

gdf_stops.to_parquet(SILVER / 'stops_gtfs.parquet')
print(f'Silver GTFS exportado: {len(gdf_stops):,} paradas')
print()
print('Por sistema:')
print(gdf_stops['sistema'].value_counts().to_string())
print()
print('Por accesibilidad:')
print(gdf_stops['accesible'].value_counts().to_string())


---
## 5. RI-6 — Infraestructura Física por Colonia (INEGI) <a id='5'></a>

**Archivo:** `infra_fisica/ri_6.shp`  
**Registros:** 1,814 polígonos · 36 columnas · **0 nulos** · CRS original: `EPSG:32614` (UTM 14N)  

El dataset ya viene en UTM 14N — no necesita reproyección. El trabajo es seleccionar variables y derivar dos nuevas normalizadas.

| Variable | Descripción | Rango |
|---|---|---|
| `AlumP_sum` | Score acumulado de alumbrado por colonia | 0 – 985 |
| `r_AlumP_Mn` | Ranking alumbrado: 1=mejor, 5=peor, 0=sin datos | 0 – 5 |
| `S_RI_6` | Score continuo de rezago de infraestructura | 19 – 43 |
| `C_RI_6` | Clase de rezago: 1=mejor condicion, 5=peor | 1 – 5 |

> 20 colonias tienen `AlumP_sum == 0` — rezago extremo. Se les asigna `vulnerabilidad_luz = 1.0`.

In [ ]:
# ── 5.1 Carga y auditoria ─────────────────────────────────────────────────
gdf_infra = gpd.read_file(INFRA_PATH)

print(f'Shape: {gdf_infra.shape}')
print(f'CRS:   {gdf_infra.crs}')
print(f'Nulos totales: {gdf_infra.isnull().sum().sum()}')
print()
print('Alumbrado:')
print(gdf_infra['AlumP_sum'].describe().round(2).to_string())
print(f'Colonias con AlumP_sum == 0: {(gdf_infra.AlumP_sum == 0).sum()}')
print()
print('r_AlumP_Mn distribucion:')
print(gdf_infra['r_AlumP_Mn'].value_counts().sort_index().to_string())
print()
print('Score compuesto:')
print(f'C_RI_6: {gdf_infra["C_RI_6"].value_counts().sort_index().to_dict()}')
print(gdf_infra['S_RI_6'].describe().round(2).to_string())


In [ ]:
# ── 5.2 Seleccion, derivacion y exportacion Silver ───────────────────────
COLS_INFRA = [
    'cve_col', 'colonia', 'alcaldia',
    'AlumP_sum', 'ALUMP_mean', 'r_AlumP_Mn',
    'P_PorcElec', 'P_AguaPot',
    'S_RI_6', 'C_RI_6',
    'geometry',
]

gdf_infra_s = gdf_infra[COLS_INFRA].copy()

# Normalizar alcaldias
gdf_infra_s['alcaldia_norm'] = (
    gdf_infra_s['alcaldia'].str.strip().str.upper().map(ALCALDIAS_MAP)
)

# Variable derivada: vulnerabilidad_luz [0.0 - 1.0]
# r_AlumP_Mn: 1=mejor->0.0, 5=peor->1.0, 0=sin datos->1.0
gdf_infra_s['vulnerabilidad_luz'] = np.where(
    gdf_infra_s['r_AlumP_Mn'] > 0,
    (gdf_infra_s['r_AlumP_Mn'] - 1) / 4,
    1.0
)

# Variable derivada: rezago_infra [0.0 - 1.0]
mn, mx = gdf_infra_s['S_RI_6'].min(), gdf_infra_s['S_RI_6'].max()
gdf_infra_s['rezago_infra'] = (gdf_infra_s['S_RI_6'] - mn) / (mx - mn)

print('Variables derivadas:')
print(f'  vulnerabilidad_luz -> min={gdf_infra_s.vulnerabilidad_luz.min():.2f}  max={gdf_infra_s.vulnerabilidad_luz.max():.2f}')
print(f'  rezago_infra       -> min={gdf_infra_s.rezago_infra.min():.2f}  max={gdf_infra_s.rezago_infra.max():.2f}')

gdf_infra_s.to_parquet(SILVER / 'infra_ri6.parquet')
gdf_infra_s.to_crs(WGS84).to_file(SILVER / 'infra_ri6_wgs84.gpkg', driver='GPKG')
print(f'Silver RI-6 exportado: {len(gdf_infra_s):,} colonias')


In [ ]:
# ── 5.3 Vista previa ──────────────────────────────────────────────────────
gdf_infra_s[['colonia','alcaldia_norm','r_AlumP_Mn','vulnerabilidad_luz','C_RI_6','rezago_infra']].head(10)


---
## 6. Unificación y Exportación — Gold Layer <a id='6'></a>

Centraliza los cuatro datasets Silver en DuckDB para consultas SQL analíticas.  
Esta capa es la entrada directa al cálculo del **Índice de Injusticia Espacial**.

In [ ]:
# ── 6.1 Inicializar DuckDB ────────────────────────────────────────────────
con = duckdb.connect(':memory:')
con.execute('INSTALL spatial; LOAD spatial;')
con.execute('INSTALL parquet;  LOAD parquet;')

# Registrar los Parquet Silver como vistas virtuales (zero-copy)
silver_str = str(SILVER)
con.execute(f"CREATE VIEW fgj   AS SELECT * FROM read_parquet('{silver_str}/fgj_2024.parquet')")
con.execute(f"CREATE VIEW wifi  AS SELECT * FROM read_parquet('{silver_str}/wifi_c5.parquet')")
con.execute(f"CREATE VIEW stops AS SELECT * FROM read_parquet('{silver_str}/stops_gtfs.parquet')")
con.execute(f"CREATE VIEW infra AS SELECT * FROM read_parquet('{silver_str}/infra_ri6.parquet')")

print('Vistas DuckDB registradas:')
print(con.execute('SHOW TABLES').df().to_string(index=False))


In [ ]:
# ── 6.2 Resumen estadistico via SQL ───────────────────────────────────────
print('=== FGJ por relevancia ===')
q_fgj = """
    SELECT
        relevancia,
        COUNT(*) AS n_registros,
        COUNT(DISTINCT mes_num) AS meses_cubiertos,
        COUNT(DISTINCT alcaldia_norm) AS alcaldias
    FROM fgj
    GROUP BY relevancia
    ORDER BY n_registros DESC
"""
print(con.execute(q_fgj).df().to_string(index=False))
print()
print('=== WiFi C5 por alcaldia ===')
q_wifi = """
    SELECT alcaldia_norm, COUNT(*) AS n_postes
    FROM wifi
    GROUP BY alcaldia_norm
    ORDER BY n_postes DESC
"""
print(con.execute(q_wifi).df().to_string(index=False))


In [ ]:
# ── 6.3 Tabla Gold: conteos por alcaldia ─────────────────────────────────
q_gold = """
    SELECT
        f.alcaldia_norm,
        COUNT(CASE WHEN f.relevancia = 'peaton'     THEN 1 END) AS delitos_peaton,
        COUNT(CASE WHEN f.relevancia = 'transporte' THEN 1 END) AS delitos_transporte,
        COUNT(*)                                                  AS delitos_total
    FROM fgj f
    WHERE f.alcaldia_norm IS NOT NULL
    GROUP BY f.alcaldia_norm
    ORDER BY delitos_peaton DESC
"""
gold_alcaldia = con.execute(q_gold).df()

gold_str = str(GOLD)
gold_alcaldia.to_parquet(f'{gold_str}/conteos_alcaldia.parquet', index=False)
print('Gold exportado: conteos_alcaldia.parquet')
gold_alcaldia


In [ ]:
# ── 6.4 Reporte final de calidad ──────────────────────────────────────────
datasets = [
    ('fgj_2024.parquet',   len(gdf_fgj),     'UTM 14N'),
    ('wifi_c5.parquet',    len(gdf_wifi),    'UTM 14N'),
    ('stops_gtfs.parquet', len(gdf_stops),   'UTM 14N'),
    ('infra_ri6.parquet',  len(gdf_infra_s), 'UTM 14N'),
]

print('REPORTE FINAL - DATA QUALITY GOLD LAYER')
print('=' * 62)
print(f'  {"Dataset":<26} {"Filas":>8}   {"CRS":<12}  Nulos')
print('  ' + '-' * 56)
for name, filas, crs in datasets:
    print(f'  {name:<26} {filas:>8,}   {crs:<12}  0')
print()
print('Transformaciones aplicadas:')
for t in [
    'Alcaldias normalizadas (16 valores canonicos)',
    'Fechas convertidas a datetime64',
    '8,997 registros FGJ sin coords descartados',
    '47 paradas GTFS fuera de bbox descartadas',
    'Estratificacion FGJ: peaton | transporte | otro',
    'Sistema de transporte decodificado desde zone_id',
    'Variables derivadas: vulnerabilidad_luz, rezago_infra',
    'Todos los GeoDataFrames en EPSG:32614 (UTM 14N)',
]:
    print(f'  OK  {t}')
print()
print('Pendiente (siguiente sprint):')
for t in [
    'Geocodificar unidades medicas con DENUE/INEGI',
    'Join espacial FGJ -> poligono colonia RI-6',
    'Analisis temporal: delitos por hora del dia',
    'Ajuste por cifra negra via INEGI ENSU',
]:
    print(f'  []  {t}')
